# Activation Patching — Zero Ablation on Attention Heads

**Question:** Which attention heads in LLaVA's language model causally contribute to object recognition?

**Method:** For each attention head (32 layers × 32 heads = 1024 total), zero out its output and measure the drop in the model's probability difference $p(\text{Yes}) - p(\text{No})$ on POPE yes/no questions. A large positive drop means the head was important for the model's original prediction.

We run this on three representative examples — TP (correctly says yes), TN (correctly says no), FP (incorrectly says yes) — to see whether different heads are recruited for different kinds of successes and failures.


In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from transformers import LlavaForConditionalGeneration, AutoProcessor
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
# Resolve paths relative to this notebook
NB_DIR = Path(os.getcwd())  # patching/
PROJECT_DIR = NB_DIR.parent

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
POPE_FILE = PROJECT_DIR / "pope_train" / "coco_train_pope_adversarial.jsonl"
IMAGE_ROOT = PROJECT_DIR / "data" / "coco2014" / "train2014" / "train2014"

NUM_HEADS = 32
NUM_LAYERS = 32
HIDDEN_DIM = 4096
HEAD_DIM = HIDDEN_DIM // NUM_HEADS  # 128

## 1. Load Model & Processor

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} ...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model.eval()

# Correct module paths:
#   LlavaForConditionalGeneration.model -> LlavaModel
#   .language_model -> LlamaModel
#   .layers[i] -> LlamaDecoderLayer
#   .self_attn -> LlamaAttention
#   .o_proj -> Linear
lm = model.model.language_model
print(f"Model loaded on {model.device}")
print(f"Language model layers: {len(lm.layers)}")
print(f"Hidden dim: {lm.config.hidden_size}")
print(f"Num attention heads: {lm.config.num_attention_heads}")

In [ ]:
# Get token IDs for "Yes" and "No" — used for probability difference p(Yes) - p(No)
yes_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
no_id = tokenizer.encode("No", add_special_tokens=False)[0]
print(f"'Yes' token id: {yes_id}, '{tokenizer.decode([yes_id])}'")
print(f"'No'  token id: {no_id}, '{tokenizer.decode([no_id])}'")


## 2. Run Inference to Find TP / TN / FP Examples

In [ ]:
# Quick peek at available results
import glob
for f in sorted(glob.glob(str(PROJECT_DIR / "results" / "train" / "*.jsonl"))):
    name = os.path.basename(f)
    n_lines = sum(1 for _ in open(f))
    print(f"  results/train/{name}  ({n_lines} answers)")

for f in sorted(glob.glob(str(PROJECT_DIR / "results" / "val" / "*.jsonl"))):
    name = os.path.basename(f)
    n_lines = sum(1 for _ in open(f))
    print(f"  results/val/{name}  ({n_lines} answers)")

In [ ]:
def build_prompt(question_text: str) -> str:
    """Build the LLaVA prompt exactly as in the POPE data — no extra instructions."""
    return (
        f"USER: <image>\n"
        f"Answer with only Yes or No. {question_text}\n"
        f"ASSISTANT:"
    )

def get_prob_diff(outputs):
    """Extract p(Yes) - p(No) from model outputs using softmax over the vocabulary.
    Positive → model prefers Yes. Negative → model prefers No."""
    logits = outputs.logits[0, -1, :]  # last token position
    probs = torch.softmax(logits.float(), dim=-1)
    return probs[yes_id].item() - probs[no_id].item()


In [ ]:
# Load existing model answers and join with labels to classify TP/TN/FP
ANS_FILE = PROJECT_DIR / "results" / "train" / "llava15_7b_adversarial_train.jsonl"
LABEL_FILE = PROJECT_DIR / "pope_train" / "coco_train_pope_adversarial.jsonl"

answers = []
with open(ANS_FILE) as f:
    for line in f:
        if line.strip():
            answers.append(json.loads(line))

labels = []
with open(LABEL_FILE) as f:
    for line in f:
        if line.strip():
            labels.append(json.loads(line))

# Index labels by question_id
label_by_qid = {l["question_id"]: l for l in labels}

results = []
for ans in answers:
    qid = ans["question_id"]
    lab = label_by_qid.get(qid)
    if lab is None:
        continue
    pred = ans["answer"]
    results.append((lab, pred, None))

print(f"Loaded {len(results)} matched (answer + label) pairs")

In [ ]:
# Classify into TP, TN, FP, FN using saved answers (no logit_diff yet)
tp = [(item, None) for item, pred, _ in results if item["label"] == "yes" and pred == "yes"]
tn = [(item, None) for item, pred, _ in results if item["label"] == "no"  and pred == "no"]
fp = [(item, None) for item, pred, _ in results if item["label"] == "no"  and pred == "yes"]
fn = [(item, None) for item, pred, _ in results if item["label"] == "yes" and pred == "no"]

print(f"TP: {len(tp)} | TN: {len(tn)} | FP: {len(fp)} | FN: {len(fn)}")

# Pick one representative of each type
tp_example, _ = tp[0]
tn_example, _ = tn[0]
fp_example, _ = fp[0]

print(f"\nTP example: {tp_example['text'][:80]} (label={tp_example['label']})")
print(f"TN example: {tn_example['text'][:80]} (label={tn_example['label']})")
print(f"FP example: {fp_example['text'][:80]} (label={fp_example['label']})")

## 3. Zero Ablation — Sweep All Heads

For each (layer, head) pair, we hook into the input of `o_proj` (the output projection after multi-head attention) and zero out the 128-dim slice corresponding to that head before it gets projected back into the residual stream.

In [ ]:
def ablate_and_measure(model, processor, item, image_root, layer_idx, head_idx):
    """
    Run a forward pass with head (layer_idx, head_idx) zero-ablated.
    Returns p(Yes) - p(No) after ablation.
    """
    start = head_idx * HEAD_DIM
    end = start + HEAD_DIM

    # model.model.language_model.layers[i].self_attn.o_proj
    o_proj = model.model.language_model.layers[layer_idx].self_attn.o_proj

    def pre_hook(module, args):
        # args[0] is input to o_proj, shape (batch, seq_len, hidden_dim)
        modified = args[0].clone()
        modified[:, :, start:end] = 0.0
        return (modified,)

    handle = o_proj.register_forward_pre_hook(pre_hook, with_kwargs=False)

    try:
        with torch.inference_mode():
            outputs = model(**inputs_cache[item["question_id"]])
        return get_prob_diff(outputs)
    finally:
        handle.remove()


def sweep_heads(model, processor, item, image_root, desc="Ablating"):
    """
    Sweep all 32×32 attention heads for one example.
    Returns (baseline_pd, effects_matrix) where effects[layer, head] = baseline - ablated.
    """
    # Compute baseline once
    with torch.inference_mode():
        outputs = model(**inputs_cache[item["question_id"]])
    baseline_pd = get_prob_diff(outputs)

    effects = np.zeros((NUM_LAYERS, NUM_HEADS))

    for layer in tqdm(range(NUM_LAYERS), desc=desc):
        for head in range(NUM_HEADS):
            ablated_pd = ablate_and_measure(model, processor, item, image_root, layer, head)
            effects[layer, head] = baseline_pd - ablated_pd

    return baseline_pd, effects


In [ ]:
# Pre-cache processed inputs for the three examples to avoid re-processing
inputs_cache = {}

for ex_name, ex_item in [("TP", tp_example), ("TN", tn_example), ("FP", fp_example)]:
    img = Image.open(IMAGE_ROOT / ex_item["image"]).convert("RGB")
    prompt = build_prompt(ex_item["text"])
    inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
    inputs_cache[ex_item["question_id"]] = inputs
    print(f"Cached inputs for {ex_name} (Q{ex_item['question_id']})")

In [ ]:
print("=" * 60)
print("Sweeping TP example (label=yes, model says yes)")
print("=" * 60)
tp_baseline, tp_effects = sweep_heads(model, processor, tp_example, IMAGE_ROOT, desc="Ablating TP")

In [ ]:
print("\n" + "=" * 60)
print("Sweeping TN example (label=no, model says no)")
print("=" * 60)
tn_baseline, tn_effects = sweep_heads(model, processor, tn_example, IMAGE_ROOT, desc="Ablating TN")

print("\n" + "=" * 60)
print("Sweeping FP example (label=no, model says yes)")
print("=" * 60)
fp_baseline, fp_effects = sweep_heads(model, processor, fp_example, IMAGE_ROOT, desc="Ablating FP")

## 4. Visualize — Indirect Effect Heatmaps (Probability Drop)

**Interpretation:**
- **Red (positive):** Ablating this head *decreases* p(Yes) − p(No), meaning the head was **promoting** the model's original prediction.
- **Blue (negative):** Ablating this head *increases* p(Yes) − p(No), meaning the head was **suppressing** the model's original prediction.
- **White (~0):** This head has negligible causal effect on this particular question.


In [ ]:
def plot_heatmap(effects, baseline_pd, title, vmin=None, vmax=None):
    """Plot a single 32×32 heatmap of indirect effects (probability drop)."""
    if vmin is None:
        vmax_abs = max(abs(effects.min()), abs(effects.max()))
        vmin, vmax = -vmax_abs, vmax_abs

    fig, ax = plt.subplots(1, 1, figsize=(8, 7))
    sns.heatmap(
        effects,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        xticklabels=range(0, 32),
        yticklabels=range(0, 32),
        cbar_kws={"label": "Probability Drop  p(Yes)−p(No)  (baseline − ablated)", "shrink": 0.8},
        linewidths=0.0,
    )
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_title(f"{title}\n(baseline p(Yes)−p(No) = {baseline_pd:.4f})")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


# Use consistent color scale across all three single-example plots
all_effects = np.stack([tp_effects, tn_effects, fp_effects])
vmax_abs = max(abs(all_effects.min()), abs(all_effects.max()))
vmin, vmax = -vmax_abs, vmax_abs

plot_heatmap(tp_effects, tp_baseline,
             f"TP: '{tp_example['text'][:60]}...'", vmin, vmax)
plot_heatmap(tn_effects, tn_baseline,
             f"TN: '{tn_example['text'][:60]}...'", vmin, vmax)
plot_heatmap(fp_effects, fp_baseline,
             f"FP: '{fp_example['text'][:60]}...'", vmin, vmax)


## 5. Interpretation

Compare the three heatmaps:

| Case | What it shows |
|------|--------------|
| **TP** (true positive) | Which heads causally drive a correct "yes" answer when the object IS present? Expect red heads — they carry visual evidence for the object. |
| **TN** (true negative) | Which heads causally drive a correct "no" answer when the object is absent? Blue heads here are pushing toward "yes" — removing them makes the model more confident in "no." |
| **FP** (false positive) | Which heads cause a spurious "yes" when the object is NOT present? Red heads here are likely driving a hallucinated or bias-driven answer. These are candidates for *editing* to reduce false positives. |

**Key things to look for:**
- Heads that show up in TP but not FP → genuinely useful for object recognition
- Heads that show up strongly in FP → potential spurious-correlation detectors
- Late-layer heads (layers 20–31) that are important across all three cases → likely involved in answer formatting rather than visual reasoning
- Early/mid-layer heads that differ between cases → likely involved in visual processing

In [ ]:
def print_top_heads(effects, label, n=8):
    """Print the top-N most influential heads (by absolute effect)."""
    flat = [(effects[l, h], l, h) for l in range(NUM_LAYERS) for h in range(NUM_HEADS)]
    flat.sort(key=lambda x: -abs(x[0]))
    print(f"\n{label} — top {n} most influential heads:")
    for i, (val, l, h) in enumerate(flat[:n]):
        direction = "promotes pred" if val > 0 else "suppresses pred"
        print(f"  {i+1}. L{l:02d}H{h:02d}  indirect_effect={val:+.3f}  ({direction})")

print_top_heads(tp_effects, "TP")
print_top_heads(tn_effects, "TN")
print_top_heads(fp_effects, "FP")

## Key Findings

- Ablation of earlier layers (0-15) primarily removes visual information — they likely project visual features into the semantic space.
- Middle layers (16-24) show mixed effects — they may translate visual features into object-specific decisions.
- Later layers (25-31) often show smaller effects, as the model has already committed to an answer.
- For FP cases, the "hallucinatory" heads are particularly interesting — they override the absence signal from visual features.

## 6. Average Across 50 Samples per Category

Single-example heatmaps can be noisy — a head might fire for one specific object but not generalize. Averaging across 50 samples per category (TP, TN, FP) gives a cleaner signal for which heads consistently matter.

⚠️ **Runtime:** ~2.5 min per example × 150 examples ≈ **6 hours**. Set `N_SAMPLES` lower (e.g., 10) for a quick pass.


In [ ]:
N_SAMPLES = 20

# Pick N_SAMPLES from each category (skip if not enough)
import random
random.seed(42)

tp_samples = random.sample(tp, min(N_SAMPLES, len(tp)))
tn_samples = random.sample(tn, min(N_SAMPLES, len(tn)))
fp_samples = random.sample(fp, min(N_SAMPLES, len(fp)))

print(f"Selected {len(tp_samples)} TP, {len(tn_samples)} TN, {len(fp_samples)} FP")

# Extend inputs_cache with all new samples
all_samples = [("TP", s[0]) for s in tp_samples] + \
              [("TN", s[0]) for s in tn_samples] + \
              [("FP", s[0]) for s in fp_samples]

for category, item in tqdm(all_samples, desc="Pre-caching inputs"):
    if item["question_id"] in inputs_cache:
        continue
    img = Image.open(IMAGE_ROOT / item["image"]).convert("RGB")
    prompt = build_prompt(item["text"])
    inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
    inputs_cache[item["question_id"]] = inputs

print(f"Inputs cache size: {len(inputs_cache)}")


In [ ]:
def sweep_multi(samples, label, model, processor, image_root):
    """
    Sweep all 32×32 heads across multiple samples.
    Returns (baselines, all_effects) where:
        baselines: list of baseline p(Yes)-p(No) per sample
        all_effects: (n_samples, 32, 32) array of indirect effects
    """
    baselines = []
    all_effects = np.zeros((len(samples), NUM_LAYERS, NUM_HEADS))

    for idx, (item, _) in enumerate(tqdm(samples, desc=f"Sweeping {label}")):
        baseline_pd, effects = sweep_heads(model, processor, item, image_root,
                                            desc=f"  {label} #{idx+1}")
        baselines.append(baseline_pd)
        all_effects[idx] = effects

    return baselines, all_effects


tp_baselines, tp_all_effects = sweep_multi(tp_samples, "TP", model, processor, IMAGE_ROOT)
tn_baselines, tn_all_effects = sweep_multi(tn_samples, "TN", model, processor, IMAGE_ROOT)
fp_baselines, fp_all_effects = sweep_multi(fp_samples, "FP", model, processor, IMAGE_ROOT)


In [ ]:
# Average across samples
tp_mean = tp_all_effects.mean(axis=0)
tn_mean = tn_all_effects.mean(axis=0)
fp_mean = fp_all_effects.mean(axis=0)

tp_mean_baseline = np.mean(tp_baselines)
tn_mean_baseline = np.mean(tn_baselines)
fp_mean_baseline = np.mean(fp_baselines)

# Consistent colour scale across all three
all_mean = np.stack([tp_mean, tn_mean, fp_mean])
vmax_abs = max(abs(all_mean.min()), abs(all_mean.max()))
vmin, vmax = -vmax_abs, vmax_abs

plot_heatmap(tp_mean, tp_mean_baseline,
             f"TP (avg over {len(tp_samples)} samples)", vmin, vmax)
plot_heatmap(tn_mean, tn_mean_baseline,
             f"TN (avg over {len(tn_samples)} samples)", vmin, vmax)
plot_heatmap(fp_mean, fp_mean_baseline,
             f"FP (avg over {len(fp_samples)} samples)", vmin, vmax)


In [ ]:
print("Averaged top heads (by absolute mean indirect effect):")
print_top_heads(tp_mean, f"TP (N={len(tp_samples)})")
print_top_heads(tn_mean, f"TN (N={len(tn_samples)})")
print_top_heads(fp_mean, f"FP (N={len(fp_samples)})")
